In [2]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, json, shutil, time
sys.path.append("/content/drive/MyDrive/tcc")
from src import config, modelos, treino

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(os.cpu_count())
print("Usando:", DISPOSITIVO)


@torch.no_grad()
def prever(modelo, X, tamanho_lote=64):
    """Probabilidade de cada crop ser IA, funcionando em GPU ou CPU."""
    modelo.eval()
    probs = []
    for i in range(0, len(X), tamanho_lote):
        xb = treino.preparar_lote(X[i:i + tamanho_lote], DISPOSITIVO)
        if DISPOSITIVO == "cuda":
            with torch.autocast("cuda", dtype=torch.float16):
                saida = modelo(xb)
        else:
            saida = modelo(xb)
        probs.append(torch.softmax(saida.float(), 1)[:, 1].cpu())
    return torch.cat(probs).numpy()


def carregar_pesos(caminho, tentativas=5):
    """Copia o arquivo do Drive para o disco do Colab e carrega, tentando de novo se o Drive falhar."""
    local = "/content/_pesos_tmp.pt"
    for t in range(tentativas):
        try:
            shutil.copy(caminho, local)
            return torch.load(local, map_location=DISPOSITIVO)
        except (OSError, SystemError, RuntimeError) as e:
            print(f"  Falha ao ler {caminho} ({e}); tentativa {t + 1}/{tentativas}")
            time.sleep(10)
    raise RuntimeError(f"Não foi possível ler {caminho}")

Mounted at /content/drive
Usando: cuda


In [3]:
ORIGEM = f"{config.ELA_DIR}/etapa1"
TESTES = {"original": "teste_original", "jpeg90": "teste_jpeg90", "firefly": "teste_firefly"}

dados_teste = {}
for chave, nome in TESTES.items():
    dados_teste[(chave, "rotulos")] = pd.read_csv(f"{ORIGEM}/{nome}_rotulos.csv")
    for entrada in ["ela", "rgb"]:
        dados_teste[(chave, entrada)] = torch.from_numpy(np.load(f"{ORIGEM}/{nome}_{entrada}.npy"))
    print(f"{chave:10s} {len(dados_teste[(chave, 'rotulos')])} crops")

original   1200 crops
jpeg90     1200 crops
firefly    600 crops


In [4]:
def por_imagem(rotulos, probs):
    """Junta os 4 crops de cada imagem tirando a média da probabilidade de ser IA."""
    d = rotulos[["id", "classe", "rotulo", "aparelho_ou_gerador"]].copy()
    d["prob_ia"] = probs
    return d.groupby(["id", "classe", "rotulo", "aparelho_ou_gerador"], as_index=False)["prob_ia"].mean()


def calcular_metricas(img):
    y = img["rotulo"].values
    p = img["prob_ia"].values
    pred = (p >= 0.5).astype(int)
    vn, fp, fn, vp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "acuracia": accuracy_score(y, pred),
        "precisao": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "auc": roc_auc_score(y, p) if len(set(y)) > 1 else np.nan,
        "vp": vp, "fn": fn, "fp": fp, "vn": vn,
    }

In [5]:
SAIDA = f"{config.RESULTADOS}/etapa1"
os.makedirs(f"{SAIDA}/predicoes", exist_ok=True)
ARQ_METRICAS = f"{SAIDA}/metricas.csv"

rodadas = sorted(
    d for d in os.listdir(config.MODELOS)
    if os.path.exists(f"{config.MODELOS}/{d}/melhor.pt") and "amostra" not in d
)
print(len(rodadas), "modelos encontrados")

linhas = pd.read_csv(ARQ_METRICAS).to_dict("records") if os.path.exists(ARQ_METRICAS) else []
feitas = {l["nome"] for l in linhas}

for nome in tqdm(rodadas):
    if nome in feitas:
        continue
    cfg = json.load(open(f"{config.MODELOS}/{nome}/config.json"))
    modelo = modelos.criar_modelo(cfg["modelo"])
    modelo.load_state_dict(carregar_pesos(f"{config.MODELOS}/{nome}/melhor.pt"))
    modelo.to(DISPOSITIVO).eval()

    imgs = {}
    for chave in ["original", "jpeg90", "firefly"]:
        probs = prever(modelo, dados_teste[(chave, cfg["entrada"])])
        imgs[chave] = por_imagem(dados_teste[(chave, "rotulos")], probs)
        imgs[chave].to_csv(f"{SAIDA}/predicoes/{nome}_{chave}.csv", index=False)

    reais_jpeg = imgs["jpeg90"][imgs["jpeg90"]["rotulo"] == 0]
    imgs["firefly_vs_real"] = pd.concat([imgs["firefly"], reais_jpeg], ignore_index=True)

    for teste, img in imgs.items():
        linhas.append({
            "nome": nome, "modelo": cfg["modelo"], "entrada": cfg["entrada"],
            "experimento": cfg["experimento"], "seed": cfg["seed"],
            "teste": teste, "n_imagens": len(img), **calcular_metricas(img),
        })

    pd.DataFrame(linhas).to_csv(ARQ_METRICAS, index=False)
    del modelo
    if DISPOSITIVO == "cuda":
        torch.cuda.empty_cache()

metricas = pd.DataFrame(linhas)
print(metricas.groupby("teste")["n_imagens"].first())

36 modelos encontrados


100%|██████████| 36/36 [00:00<00:00, 271573.64it/s]

teste
firefly            150
firefly_vs_real    300
jpeg90             300
original           300
Name: n_imagens, dtype: int64


In [6]:
metricas = pd.read_csv(ARQ_METRICAS)
ORDEM_TESTES = ["original", "jpeg90", "firefly", "firefly_vs_real"]

def media_desvio(serie):
    return f"{100 * serie.mean():.1f} ± {100 * serie.std():.1f}"

tabelas = {}
for metrica in ["acuracia", "f1", "auc"]:
    tab = (metricas.groupby(["modelo", "entrada", "experimento", "teste"])[metrica]
           .apply(media_desvio).unstack("teste"))
    tab = tab[[t for t in ORDEM_TESTES if t in tab.columns]]
    if metrica == "auc":
        tab = tab.drop(columns="firefly", errors="ignore")
    tab.to_csv(f"{SAIDA}/tabela_{metrica}.csv")
    tabelas[metrica] = tab

print("ACURÁCIA POR IMAGEM (%)")
display(tabelas["acuracia"])

ACURÁCIA POR IMAGEM (%)


teste                              original      jpeg90      firefly  \
modelo     entrada experimento                                         
cnn_artigo ela     jpeg90        50.7 ± 0.9  96.9 ± 0.5    7.6 ± 4.7   
                   misto         99.0 ± 0.3  97.0 ± 0.7    8.4 ± 2.7   
                   png           99.1 ± 0.2  53.3 ± 3.2    0.0 ± 0.0   
           rgb     jpeg90        93.1 ± 1.0  93.6 ± 0.2   59.1 ± 0.4   
                   misto         94.6 ± 1.0  94.7 ± 1.2   58.9 ± 2.7   
                   png           94.6 ± 0.5  95.1 ± 1.2   62.4 ± 2.0   
resnet18   ela     jpeg90        53.6 ± 3.7  99.1 ± 0.2   20.9 ± 8.7   
                   misto         99.6 ± 0.2  98.8 ± 0.7   28.9 ± 3.2   
                   png           99.3 ± 0.0  67.8 ± 2.0   25.6 ± 6.9   
           rgb     jpeg90        99.8 ± 0.4  99.9 ± 0.2   65.1 ± 3.4   
                   misto         99.7 ± 0.6  99.6 ± 0.5  64.9 ± 13.0   
                   png          100.0 ± 0.0  94.4 ± 2.5   89.6 ± 3.4   

teste                          firefly_vs_real  
modelo     entrada experimento                  
cnn_artigo ela     jpeg90           51.9 ± 2.9  
                   misto            52.9 ± 1.7  
                   png              50.0 ± 0.0  
           rgb     jpeg90           78.8 ± 0.2  
                   misto            79.0 ± 1.2  
                   png              80.2 ± 1.0  
resnet18   ela     jpeg90           59.8 ± 4.1  
                   misto            63.4 ± 2.5  
                   png              57.9 ± 2.0  
           rgb     jpeg90           82.4 ± 1.7  
                   misto            82.2 ± 6.6  
                   png              89.2 ± 4.0

In [7]:
deteccao = []
for nome in metricas.query("experimento == 'png'")["nome"].unique():
    cfg = json.load(open(f"{config.MODELOS}/{nome}/config.json"))
    for chave in ["original", "firefly"]:
        img = pd.read_csv(f"{SAIDA}/predicoes/{nome}_{chave}.csv")
        ia = img[img["rotulo"] == 1]
        for gerador, g in ia.groupby("aparelho_ou_gerador"):
            deteccao.append({"modelo": f"{cfg['modelo']}_{cfg['entrada']}", "gerador": gerador,
                             "deteccao": (g["prob_ia"] >= 0.5).mean()})

deteccao = (pd.DataFrame(deteccao).groupby(["gerador", "modelo"])["deteccao"]
            .mean().unstack("modelo").mul(100).round(1))
deteccao.to_csv(f"{SAIDA}/deteccao_por_gerador.csv")
display(deteccao)

modelo,cnn_artigo_ela,cnn_artigo_rgb,resnet18_ela,resnet18_rgb
gerador,,,,
dalle2,100.0,82.5,100.0,100.0
dalle3,100.0,93.7,100.0,100.0
firefly,0.0,62.4,25.6,89.6
midjourney-v5,97.0,75.8,95.5,100.0
stable-diffusion-1-3,100.0,98.5,100.0,100.0
stable-diffusion-1-4,100.0,100.0,100.0,100.0
stable-diffusion-2,100.0,100.0,100.0,100.0
stable-diffusion-xl,100.0,85.7,100.0,100.0
